# ScanOps 보안 모델 — 3B 학습 + OWASP 평가 (v11, 2026 신규 CVE 포함)

Qwen2.5-Coder-**3B** + QLoRA(4bit) + v11 데이터(OWASP Java + **2026년 5~6월 신규 NVD CVE 50건** + 안전).
v10에서 모델이 구분을 시작했고(오탐률 27%), v11은 탐지 지향(안전 44%) + 4epoch + 2026 CVE로 재현율을 끌어올린다.

**준비:** 런타임 → 런타임 유형 변경 → **T4 GPU**

**업로드 파일 2개** (로컬 `scanops-model/data/`): `lora_train_v11.jsonl`, `owasp_holdout_eval.json`

순서: ① 설치 → ② 업로드 → ③ 학습(~45분) → ④ 평가

OOM 나면: 런타임 → 세션 다시 시작 후 ①부터 다시.

## ① 의존성 설치 + GPU 확인

In [ ]:
!pip -q install transformers peft datasets accelerate bitsandbytes matplotlib
import torch
assert torch.cuda.is_available(), '런타임 유형을 T4 GPU로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0))

## ② 파일 업로드 (lora_train_v11.jsonl + owasp_holdout_eval.json)

In [ ]:
import os
for f in ['lora_train_v11.jsonl', 'owasp_holdout_eval.json']:
    if os.path.exists(f): os.remove(f)
from google.colab import files
files.upload()   # 두 파일 모두 선택
print('학습:', os.path.exists('lora_train_v11.jsonl'), '| 평가:', os.path.exists('owasp_holdout_eval.json'))

## ③ 3B QLoRA 학습 (~45분, 4 epoch, 학습 중 평가 끔 → OOM 방지)

In [ ]:
import torch, gc
try: del trainer, model
except: pass
gc.collect(); torch.cuda.empty_cache()

import json, time, os
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, Trainer, TrainingArguments)

TRAIN = 'lora_train_v11.jsonl'
HOLD  = 'owasp_holdout_eval.json'
print('파일 존재?', os.path.exists(TRAIN), os.path.exists(HOLD))

BASE = 'Qwen/Qwen2.5-Coder-3B-Instruct'
MAXLEN, R, ALPHA, EPOCHS, LR = 768, 32, 64, 4, 1e-4    # ★ EPOCHS 4 (탐지력↑)
SYS = 'You are a security code analyzer.'

rows = [json.loads(l) for l in open(TRAIN) if l.strip()]
print(f"{len(rows)}개, 안전 {100*sum(1 for r in rows if r['completion'].upper().startswith('VULNERABILITY: NONE'))/len(rows):.1f}%")
ds = Dataset.from_list(rows).train_test_split(test_size=0.1, seed=42)
tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
def fmt(e): return {'text': f'<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\n'+e['prompt']+'<|im_end|>\n<|im_start|>assistant\n'+e['completion']+'<|im_end|>'}
def tk(e):
    o=tok(e['text'],truncation=True,max_length=MAXLEN); o['labels']=o['input_ids'].copy(); return o
tr = ds['train'].map(fmt).map(tk, remove_columns=ds['train'].column_names+['text'])

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj'], task_type=TaskType.CAUSAL_LM))
model.print_trainable_parameters()

args = TrainingArguments(output_dir='out', num_train_epochs=EPOCHS, per_device_train_batch_size=1,
    per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=LR,
    lr_scheduler_type='cosine', warmup_ratio=0.03, fp16=True, logging_steps=10,
    eval_strategy='no', save_strategy='no', report_to='none',
    gradient_checkpointing=True, optim='paged_adamw_8bit')
trainer = Trainer(model=model, args=args, train_dataset=tr,
    data_collator=DataCollatorForSeq2Seq(tok, pad_to_multiple_of=8, return_tensors='pt'))
t0=time.time(); trainer.train(); print(f'학습 {(time.time()-t0)/60:.1f}분')
model.save_pretrained('adapter'); tok.save_pretrained('adapter')
LOSSLOG=[{'step':e['step'],'loss':e.get('loss')} for e in trainer.state.log_history if 'loss' in e]
json.dump(LOSSLOG, open('adapter/train_loss.json','w'), indent=2)

## ④ OWASP 외부 벤치마크 평가 (진단 5줄 → 전체 + 카테고리별)

In [ ]:
import re, json, torch
model.eval(); model.config.use_cache = True
try: model.gradient_checkpointing_disable()
except: pass
cases = json.load(open(HOLD))

def gen(c):
    msgs = [{'role':'system','content':SYS},{'role':'user','content':c['prompt']}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt', return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=60, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)

print('=== 진단 (안전3 + 취약2: 안전은 NONE, 취약은 취약점명) ===')
for c in [x for x in cases if x['label']=='safe'][:3] + [x for x in cases if x['label']=='vuln'][:2]:
    print(f"[{c['label']:4}/{c['category']:11}] {repr(gen(c)[:70])}")

def vuln_of(t): m=re.search(r'VULNERABILITY:\s*(.+)', t); return m.group(1).strip() if m else ''
def is_safe(v): return (not v) or v.upper().startswith('NONE')
CAT_KW={'sqli':['sql','89'],'xss':['xss','cross-site','scripting','79','80'],'cmdi':['command','78','77'],
 'pathtraver':['traversal','22','23','36'],'crypto':['crypto','encrypt','327','cipher',' des','broken'],
 'hash':['hash','md5','sha-1','sha1','328','326'],'ldapi':['ldap','90'],'xpathi':['xpath','643'],
 'trustbound':['trust bound','501'],'securecookie':['cookie','secure flag','614','311','315','1004'],
 'weakrand':['random','330','338','predictable']}
def cwe_ok(v,cat): t=v.lower(); return any(k in t for k in CAT_KW.get(cat,[]))

tp=fn=fp=tn=cwe=0; detail=[]; bycat={}
for i,c in enumerate(cases):
    v=vuln_of(gen(c)); fl=not is_safe(v); isv=c['label']=='vuln'
    if isv and fl: tp+=1; cwe+=cwe_ok(v,c['category'])
    elif isv and not fl: fn+=1
    elif not isv and fl: fp+=1
    else: tn+=1
    b=bycat.setdefault(c['category'],{'n':0,'ok':0}); b['n']+=1; b['ok']+=int(fl==isv)
    detail.append({'id':c['id'],'category':c['category'],'label':c['label'],'pred':v})
    if i%20==0: print(f'  {i}/{len(cases)}')
rec=tp/(tp+fn)*100; fpr=fp/(fp+tn)*100; prec=tp/(tp+fp)*100 if tp+fp else 0
acc=(tp+tn)/len(cases)*100; f1=2*prec*rec/(prec+rec) if prec+rec else 0
json.dump(detail, open('owasp_eval_detail.json','w'), ensure_ascii=False, indent=2)
print('='*62)
print(f'ScanOps 3B-v11: F1={f1:.1f} 재현율={rec:.1f}% 오탐률={fpr:.1f}% 정확도={acc:.1f}% CWE정확={cwe/(tp+fn)*100:.1f}% (TP{tp} FN{fn} FP{fp} TN{tn})')
print('--- 카테고리별 정확도 ---')
for c in sorted(bycat): print(f"  {c:13} {bycat[c]['ok']}/{bycat[c]['n']} = {100*bycat[c]['ok']/bycat[c]['n']:.0f}%")
print('Grok-3-mini: F1=62.9 재현율=60% 오탐률=30.9% CWE정확=60%')
print('='*62)

## ⑤ 학습곡선 + 어댑터 다운로드 (결과 좋으면)

In [ ]:
import matplotlib.pyplot as plt, json
L=json.load(open('adapter/train_loss.json'))
pts=[(e['step'],e['loss']) for e in L if e.get('loss') is not None]
plt.figure(figsize=(7,4)); plt.plot(*zip(*pts),'-o',ms=3)
plt.xlabel('step'); plt.ylabel('train loss'); plt.grid(alpha=.4); plt.title('learning curve (3B v11)')
plt.savefig('learning_curve_v11.png',dpi=150,bbox_inches='tight'); plt.show()
!cd adapter && zip -r ../adapter_3b_v11.zip . >/dev/null
from google.colab import files
files.download('adapter_3b_v11.zip')